# Phase 1: Basic Cleaning


In [28]:
# 1.1 import raw data
import pandas as pd
import numpy as np

df = pd.read_csv(r'D:\chennai1-pg-price-predictor\Data\raw\chennai_pg_dataset.csv')

In [29]:
df.shape

(1661, 39)

In [30]:
# 1.2 Deduplication
before = df.shape[0]

# id + occupancy combination vachu dedup pannuthu, full row vachu illa
df = df.drop_duplicates(subset=['id', 'occupancy'], keep='first')

after = df.shape[0]
print(f"{before - after} duplicate rows removed")
print("Shape after dedup:", df.shape)


130 duplicate rows removed
Shape after dedup: (1531, 39)


In [31]:
# Drop Unnecessary Columns
drop_cols = [
    'id', 'title', 'address', 'total_bathrooms',
    'gate_closing_time', 'warden', 'cooking_allowed',
    'guardian_required', 'nonveg_allowed', 'smoking_allowed'
]

existing = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=existing)

print(f"Dropped columns: {existing}")
print("Shape after dropping columns:", df.shape)

Dropped columns: ['id', 'title', 'address', 'total_bathrooms', 'gate_closing_time', 'warden', 'cooking_allowed', 'guardian_required', 'nonveg_allowed', 'smoking_allowed']
Shape after dropping columns: (1531, 29)


In [32]:
# 1.4 Drop Redundant Food Columns
food_cols = ['breakfast', 'lunch', 'dinner']

existing = [c for c in food_cols if c in df.columns]
df = df.drop(columns=existing)

print(f"Dropped columns: {existing}")
print("Shape after dropping food columns:", df.shape)

Dropped columns: ['breakfast', 'lunch', 'dinner']
Shape after dropping food columns: (1531, 26)


In [33]:
# remove rpws
df = df.dropna(subset=['rent', 'deposit', 'occupancy', 'attached_bathroom'])
df = df[df['rent'] >= 1000]

df.shape


(1436, 26)

In [34]:
# Fix data types
amenity_cols = [
    'attached_bathroom', 'mess', 'wifi', 'laundry', 'power_backup',
    'refrigerator', 'common_tv', 'room_cleaning', 'room_ac',
    'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding',
    'room_attached_bath'
]

# Check unique values before converting (safety check)
for col in amenity_cols:
    print(col, df[col].unique())


attached_bathroom [False True]
mess [nan False True]
wifi [nan True False]
laundry [nan True False]
power_backup [nan False True]
refrigerator [nan True False]
common_tv [nan True False]
room_cleaning [nan True False]
room_ac [False True nan]
room_cupboard [False True nan]
room_tv [False True nan]
room_geyser [False True nan]
room_bedding [False True nan]
room_attached_bath [False True nan]


In [35]:
# Missing values-a False nu fill pannunga (amenity available illa nu treat pannuthu)
df[amenity_cols] = df[amenity_cols].fillna(False)

# dtype-a bool aa convert pannunga
df[amenity_cols] = df[amenity_cols].astype(bool)

print("Shape after fixing amenity columns:", df.shape)
df[amenity_cols].dtypes

Shape after fixing amenity columns: (1436, 26)


C:\Users\phari\AppData\Local\Temp\ipykernel_13832\549647577.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[amenity_cols] = df[amenity_cols].fillna(False)


attached_bathroom     bool
mess                  bool
wifi                  bool
laundry               bool
power_backup          bool
refrigerator          bool
common_tv             bool
room_cleaning         bool
room_ac               bool
room_cupboard         bool
room_tv               bool
room_geyser           bool
room_bedding          bool
room_attached_bath    bool
dtype: object

In [36]:
# 1.7 — Replace -10 sentinel with NaN
df['transit_score'] = df['transit_score'].replace(-10, np.nan)

# Step Create a missingness indicator (before filling)
df['transit_score_missing'] = df['transit_score'].isna()

#  Fill missing values using locality-level median
df['transit_score'] = df.groupby('locality')['transit_score'].transform(
    lambda x: x.fillna(x.median())
)

overall_median = df['transit_score'].median()
df['transit_score'] = df['transit_score'].fillna(overall_median)

print("Remaining missing transit_score:", df['transit_score'].isna().sum())
print("transit_score_missing counts:\n", df['transit_score_missing'].value_counts())

Remaining missing transit_score: 0
transit_score_missing counts:
 transit_score_missing
False    759
True     677
Name: count, dtype: int64


d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [37]:
# 1.8 Create a missingness indicator (before filling)
df['lifestyle_score_missing'] = df['lifestyle_score'].isna()


df['lifestyle_score'] = df.groupby('locality')['lifestyle_score'].transform(
    lambda x: x.fillna(x.median())
)

overall_median_lifestyle = df['lifestyle_score'].median()
df['lifestyle_score'] = df['lifestyle_score'].fillna(overall_median_lifestyle)

print("Remaining missing lifestyle_score:", df['lifestyle_score'].isna().sum())
print("lifestyle_score_missing counts:\n", df['lifestyle_score_missing'].value_counts())

Remaining missing lifestyle_score: 0
lifestyle_score_missing counts:
 lifestyle_score_missing
False    762
True     674
Name: count, dtype: int64


d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [38]:
# Missing parking values-a 'none' category aa fill pannunga
df['parking'] = df['parking'].fillna('none')

print("parking value counts:\n", df['parking'].value_counts())

parking value counts:
 parking
Bike            1156
Bike and Car     137
none             112
Car               31
Name: count, dtype: int64


In [39]:
df['available_for'] = df['available_for'].replace('Both', 'Anyone')
print(df['available_for'].value_counts())

available_for
Anyone                  1307
Working Professional     121
Student                    8
Name: count, dtype: int64


In [40]:
# Final check
print("Final shape:", df.shape)
print("\nMissing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicates:", df.duplicated().sum())

Final shape: (1436, 28)

Missing values:
 Series([], dtype: int64)

Duplicates: 0


# Phase 2: Preprocessing

## Split the data

In [41]:
from sklearn.model_selection import train_test_split

X = df.drop('rent', axis=1)
y = df['rent']

## Train Data

In [43]:
#First 80% Train + 20% Temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [44]:
# Then temporary 20%-a 50/50 split pannuvom
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)
